In [0]:
WITH params AS (
  SELECT 30 AS p_window_days
),
normalized AS (
  SELECT
    source_version_id,
    try_to_timestamp(ingested_at_utc) AS ingested_ts_utc
  FROM 3dt2ndteam5.cwe.cwe_weaknesses
),
scoped AS (
  SELECT
    source_version_id,
    ingested_ts_utc
  FROM normalized
  CROSS JOIN params p
  WHERE ingested_ts_utc IS NOT NULL
    AND ingested_ts_utc >= timestampadd(DAY, -p.p_window_days, current_timestamp())
)
SELECT
  date(ingested_ts_utc) AS ingest_date_utc,
  date(from_utc_timestamp(ingested_ts_utc, 'Asia/Seoul')) AS ingest_date_kst,
  COUNT(*) AS ingested_rows,
  COUNT(DISTINCT source_version_id) AS distinct_source_versions,
  MAX(ingested_ts_utc) AS latest_ingest_time_utc,
  from_utc_timestamp(MAX(ingested_ts_utc), 'Asia/Seoul') AS latest_ingest_time_kst
FROM scoped
GROUP BY
  date(ingested_ts_utc),
  date(from_utc_timestamp(ingested_ts_utc, 'Asia/Seoul'))
ORDER BY ingest_date_utc;
